# `bz-ggml`: Turnkey Dual Tesla T4 Inference on Kaggle

High-performance C++ inference engine for **Breeze-TTS-2** powered by **GGML** on Dual NVIDIA Tesla T4 GPUs.

### ⚡ Fast-Start Highlights:
- **Modular Precision**: Easily switch between **FP16** (highest studio fidelity) and **Q8_0** (fast 8-bit quantized) with a single toggle.
- **Zero Wait Time**: Uses precompiled CUDA `sm_75` binaries (`breeze-cuda-sm75.tar.gz`) from GitHub. Startup takes **< 15 seconds** instead of compiling for 30 minutes!
- **Unlocked 2,040 Context**: Long-form speech generation up to ~2 minutes 10 seconds in a single pass.
- **Dual Architecture Choices**:
  1. **Option 1 (Headless Production Engine)**: Clean, zero-UI dual-GPU cluster inference for batch and microservices.
  2. **Option 2 (Gradio Studio UI)**: Interactive web studio with real-time 32-frame streaming playback, seekable master waveform, and voice persona management.

## 1. Hardware Verification
Confirm two NVIDIA Tesla T4 GPUs (16 GB VRAM each) are active.

In [ ]:
!nvidia-smi

## 2. Fast Turnkey Setup (< 15-35 Seconds Total)
Clones the repo, downloads precompiled CUDA binaries from GitHub, and pulls the selected model precision from Hugging Face.

In [ ]:
import os, subprocess, time, glob

t0 = time.time()

# ----------------------------------------------------------------------
# 🎯 MODEL PRECISION SELECTOR (Uncomment the one you want to use)
# ----------------------------------------------------------------------
MODEL_NAME = "breeze-tts-2-f16.gguf"    # 5.85 GB (Highest Studio Fidelity, Unquantized FP16)
# MODEL_NAME = "breeze-tts-2-q8_0.gguf"  # 3.32 GB (Fast 8-bit Quantized)
# ----------------------------------------------------------------------

print("🚀 [1/3] Symlinking CUDA driver...")
subprocess.run("ln -sf /usr/local/nvidia/lib64/libcuda.so /usr/lib/x86_64-linux-gnu/libcuda.so && ldconfig", shell=True)

# 1. Clone repository
if not os.path.exists("/kaggle/working/bz-ggml"):
    print("📦 [2/3] Cloning bz-ggml engine...")
    subprocess.run("git clone -b main https://github.com/sharjeel103/bz-ggml.git /kaggle/working/bz-ggml", shell=True)
else:
    print("📦 [2/3] Updating bz-ggml engine to latest commit...")
    subprocess.run("git -C /kaggle/working/bz-ggml fetch origin main && git -C /kaggle/working/bz-ggml checkout main && git -C /kaggle/working/bz-ggml pull origin main", shell=True)

# 2. Download precompiled CUDA sm_75 binaries directly from GitHub (~39 MB)
os.makedirs("/kaggle/working/bz-ggml/build", exist_ok=True)
bin_tar = "/kaggle/working/bz-ggml/breeze-cuda-sm75.tar.gz"
breeze_so = "/kaggle/working/bz-ggml/build/libbreeze.so"
if not os.path.exists(breeze_so):
    print("⚡ [3/3] Downloading precompiled CUDA binaries from GitHub (~39 MB in 2 seconds)...")
    BIN_URL = "https://github.com/sharjeel103/bz-ggml/raw/prebuilt-t4/breeze-cuda-sm75.tar.gz"
    subprocess.run(f"wget -q -c {BIN_URL} -O {bin_tar}", shell=True)
    subprocess.run(f"tar -xzf {bin_tar} -C /kaggle/working/bz-ggml/", shell=True)
    subprocess.run("cp -P /kaggle/working/bz-ggml/breeze-cuda-sm75/lib/* /kaggle/working/bz-ggml/build/", shell=True)

# 3. Download selected model from Hugging Face
os.makedirs("/kaggle/working/models", exist_ok=True)
model_path = os.path.join("/kaggle/working/models", MODEL_NAME)
if not os.path.exists(model_path):
    print(f"📥 Downloading {MODEL_NAME} from Hugging Face...")
    from huggingface_hub import hf_hub_download
    hf_hub_download(repo_id="HoppouAI/Breeze-TTS-2.cpp", filename=MODEL_NAME, local_dir="/kaggle/working/models")

# Save active model path for automatic discovery by Server and Gradio cells
with open("/kaggle/working/models/active_model.txt", "w") as f:
    f.write(model_path)

print()
print(f"✅ Turnkey setup ready in {time.time()-t0:.1f}s! Active model: {MODEL_NAME}")


---
## 🌟 Code Type 1: Headless Production Server / Engine (Zero Gradio)
Run pure dual-GPU cluster inference with zero UI dependencies. Automatically uses whichever model was downloaded.

In [ ]:
import sys, os, time, glob
sys.path.insert(0, "/kaggle/working/bz-ggml/python")
from bz_ggml import DualInstanceCluster, EngineOrchestrator, Parcel, TaskType, TaskSpec
from IPython.display import Audio, display

# Auto-detect active downloaded model (modular: works with FP16, Q8, or any downloaded model)
model_path = None
if os.path.exists("/kaggle/working/models/active_model.txt"):
    with open("/kaggle/working/models/active_model.txt") as f:
        model_path = f.read().strip()
if not model_path or not os.path.exists(model_path):
    models = sorted(glob.glob("/kaggle/working/models/*.gguf"))
    if not models:
        raise FileNotFoundError("No .gguf model found in /kaggle/working/models. Run Cell 2 first!")
    model_path = models[-1]

print(f"🚀 Initializing DualInstanceCluster with: {os.path.basename(model_path)}")

# 1. Initialize Dual-GPU Cluster across GPU 0 and GPU 1
cluster = DualInstanceCluster(
    model_path=model_path,
    lib_path="/kaggle/working/bz-ggml/build/libbreeze.so",
    enable_vocoder=True,
    vocoder_mode="on"
)

# 2. Initialize Pure Decoupled Orchestrator (Dedicated queues & event-driven non-blocking router)
orchestrator = EngineOrchestrator(cluster=cluster)

# 3. Create Full Multi-Stage Production Parcel (TTS + Resemble Enhance + Whisper ASR + Studio Master)
parcel = Parcel(
    parcel_id="kaggle_e2e_demo",
    tasks=[
        TaskSpec(TaskType.TTS),
        TaskSpec(TaskType.ENHANCE, {"chunk_seconds": 10.0}),
        TaskSpec(TaskType.ASR, {"output_format": "txt"}),
        TaskSpec(TaskType.MASTER)
    ],
    payload={
        "script_text": "The production inference engine is fully operational on dual Tesla T4 GPUs with decoupled stage queues and zero Gradio overhead.",
        "instruction": "Speak with natural clarity and professional authority."
    },
    return_artifacts=["audio_wav", "audio_mp3", "subtitles_srt", "transcript_txt"]
)

def progress_logger(stage, pct, msg):
    print(f"  [{stage.upper():<7}] {pct*100:5.1f}% | {msg}")

print("🎙️ Dispatching multi-stage Parcel via Decoupled Engine Orchestrator...")
t0 = time.time()
res = orchestrator.dispatch(parcel, progress_callback=progress_logger)
total_elapsed = time.time() - t0

print()
print("🎉 Parcel Pipeline Completed Successfully!")
print(f"  • Model:        {os.path.basename(model_path)}")
print(f"  • Audio Time:   {res.duration_s:.2f}s ({res.chunks_count} chunk(s))")
print(f"  • Total Latency: {total_elapsed:.2f}s (RTF: {total_elapsed/max(0.1, res.duration_s):.3f})")
print(f"  • Stage Timings: {res.stage_timings}")
if res.transcript:
    print(f"  • Verbatim ASR:  "{res.transcript}"")

# Play generated studio audio
if res.audio_pcm is not None:
    from bz_ggml.audio_pipeline import encode_pcm_to_wav_bytes
    wav_bytes = encode_pcm_to_wav_bytes(res.audio_pcm, res.sample_rate)
    display(Audio(wav_bytes, rate=res.sample_rate))

# Clean up cluster resources to free GPU VRAM for Cell 8 (Gradio Studio)
cluster.close()
del cluster, orchestrator
import gc; gc.collect()
print("✨ Engine closed cleanly. VRAM freed for Gradio!")


---
## 🌟 Code Type 2: Interactive Gradio Studio (Web UI + Public Link)
Launch the full studio UI on port 7860 with live streaming, voice cloning, and public tunneling. Automatically uses whichever model was downloaded.

In [ ]:
import subprocess, time, os, glob

# Auto-detect active downloaded model (modular: works with FP16, Q8, or any downloaded model)
model_path = None
if os.path.exists("/kaggle/working/models/active_model.txt"):
    with open("/kaggle/working/models/active_model.txt") as f:
        model_path = f.read().strip()
if not model_path or not os.path.exists(model_path):
    models = sorted(glob.glob("/kaggle/working/models/*.gguf"))
    if not models:
        raise FileNotFoundError("No .gguf model found in /kaggle/working/models. Run Cell 2 first!")
    model_path = models[-1]

# Install optional whisper ASR & Gradio
subprocess.run('pip install -q "av<18" faster-whisper soundfile deepspeed hjson && pip install -q --no-deps resemble-enhance gradio', shell=True)
os.makedirs("/kaggle/working/data/voices", exist_ok=True)

# Kill any existing processes
subprocess.run("pkill -9 -f gradio_app.py || true; pkill -9 -f frpc || true", shell=True)
time.sleep(2)

env_vars = (
    "export PYTHONPATH=/kaggle/working/bz-ggml/python: && "
    "export PYTHONUNBUFFERED=1 && "
)

cmd = (
    env_vars +
    f"nohup python3 -u /kaggle/working/bz-ggml/python/gradio_app.py "
    f"--model {model_path} "
    f"--lib /kaggle/working/bz-ggml/build/libbreeze.so "
    f"--vocoder-mode on "
    f"--voices-dir /kaggle/working/data/voices "
    f"--asr-device 0 "
    f"--share --port 7860 > /kaggle/working/gradio_studio.log 2>&1 &"
)

subprocess.run(cmd, shell=True, executable="/bin/bash")
print(f"🚀 Gradio Studio launched with {os.path.basename(model_path)} on port 7860!")
print("⏳ Waiting for public Gradio tunnel URL...")
print()

public_url = None
for _ in range(60):
    time.sleep(2)
    if os.path.exists("/kaggle/working/gradio_studio.log"):
        with open("/kaggle/working/gradio_studio.log", "r") as f:
            for line in f.read().splitlines():
                if "Running on public URL:" in line:
                    public_url = line.split("Running on public URL:")[1].strip()
                    break
    if public_url:
        break

print("=" * 65)
print(f"  🌟 PUBLIC GRADIO LINK ({os.path.basename(model_path)}):  {public_url}")
print("=" * 65)
